# 05. Lecke - Agentikus RAG


## Beállítás

Ez a jegyzetfüzet az Agentic RAG (Retrieval-Augmented Generation) mintát mutatja be a Microsoft Agent Framework használatával.

**Előfeltételek:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — az Azure AI Search szolgáltatásod végpontja
- `AZURE_SEARCH_API_KEY` — az Azure AI Search API kulcsod
- Azure OpenAI telepítés környezeti változókon keresztül konfigurálva
- Azure CLI bejelentkezve (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Mi az az Agentic RAG?

A hagyományos RAG egy rögzített folyamatot követ: dokumentumokat lekérdez, majd választ generál. Az **Agentic RAG** tovább megy, mivel az ágense önállóságot kap arra, hogy eldöntse, **mikor** és **hogyan** szerezzen be információkat.

Az Agentic RAG segítségével az ágens képes:
- **Dönteni** arról, hogy szükséges-e lekérdezés válaszadás előtt
- **Kiválasztani**, melyik adatforrást vagy eszközt kérdezze le
- **Értékelni** a lekérdezett eredményeket, és ha az első próbálkozás nem elegendő, további lekérdezéseket végrehajtani
- **Összegezni** az információkat több lekérdezési lépésből egy koherens válaszba

Ez rugalmasságot és pontosságot ad az ágensnek a statikus lekérdezés-utáni generáláshoz képest.


## Keresőeszköz létrehozása

Az Agentic RAG-ben a külső adatforrások úgy vannak becsomagolva, hogy **eszközökként** szolgálnak, amelyeket az ügynök igény szerint meghívhat. Ez lehetővé teszi, hogy az ügynök a lekérdezést csak egy további műveletként kezelje, nem pedig kötelező lépésként. 

Az alábbiakban definiálunk egy utazási tudástárat, és eszközként tesszük elérhetővé, amelyet az ügynök a célállomás információinak lekérdezésére hívhat meg.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## A RAG ügynök létrehozása

Most létrehozunk egy olyan ügynököt, akit arra utasítanak, hogy **mindig információt szerezzen be a válaszadás előtt**. Az ügynök a `search_travel_knowledge` eszközt használja, hogy válaszait a tudásalapra alapozza, ahelyett, hogy a saját tanulási adatbázisára támaszkodna.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## Iteratív lekérdezés — A Maker-Checker minta

Az Agentic RAG egyik kulcsfontosságú előnye az **iteratív lekérdezés**. Az ügynök több lekérdezési kört is végrehajthat, hogy ellenőrizze, finomítsa vagy kiegészítse a kezdeti eredményeit — hasonlóan a "maker-checker" munkafolyamathoz:

1. **Maker lépés**: Az ügynök lekéri a kezdeti információkat, és megfogalmaz egy választ.
2. **Checker lépés**: Az ügynök további lekérdezéseket végez a részletek ellenőrzésére vagy a hiányok pótlására.

Alább az ügynök egy olyan kérdést kap, amely több úti célt hasonlít össze, és ez arra készteti, hogy többször is keressen.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## Összefoglaló

Ebben a leckében megtanultad, hogyan építs fel egy **Agentic RAG** rendszert a Microsoft Agent Framework segítségével:

- Az **Agentic RAG** lehetővé teszi az ügynökök számára, hogy önállóan döntsenek az információk lekérésének idejéről, így a lekérés dinamikus, nem pedig rögzített lesz.
- **Eszközök adatforrásként**: Külső tudásbázisok (például az Azure AI Search) eszközként vannak becsomagolva, amelyet az ügynök meghívhat.
- **Ismétlődő lekérés**: A maker-checker minta lehetővé teszi az ügynök számára, hogy többször is lekérdezzen — keresés, ellenőrzés és finomítás — mielőtt végleges választ adna.

Éles környezetben az in-memory `TRAVEL_KNOWLEDGE_BASE` helyett valódi Azure AI Search indexet használnál a nagy volumenű utazási dokumentumok lekéréséhez.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Figyelmeztetés**:  
Ezt a dokumentumot az [Co-op Translator](https://github.com/Azure/co-op-translator) AI fordítószolgáltatás segítségével fordítottuk. Bár a pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti, anyanyelvű dokumentum tekintendő hiteles forrásnak. Kritikus információk esetén professzionális emberi fordítást javaslunk. Nem vállalunk felelősséget a fordítás használatából eredő félreértésekért vagy téves értelmezésekért.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
